In [1]:
# Install the required libraries
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu beautifulsoup4 docx2txt scikit-learn pymupdf pymupdf4llm

# Install Tesseract for OCR in Colab
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr


import os
import re

from getpass import getpass
from google.colab import files

from langchain_community.document_loaders import (
    WebBaseLoader,
    Docx2txtLoader,
    TextLoader
)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.vectorstores import FAISS

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pymupdf4llm


# Add your Gemini API key
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass(
        "Enter your Google API key: "
    )


# Gemini models
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)


# These will be rebuilt whenever a new source is loaded
chunks = []
vector_store = None
tfidf = None
tfidf_matrix = None


def clean_text(text):
    """Remove unnecessary whitespace from extracted text."""
    return re.sub(r"\s+", " ", text).strip()


def load_pdf(file_path):
    """
    Extract a PDF using PyMuPDF4LLM.

    OCR is automatic. Text-based pages are extracted normally,
    while scanned/image-based pages can be processed with OCR.
    """

    print("\nProcessing PDF...")
    print("Layout extraction + automatic OCR enabled.")

    try:
        page_data = pymupdf4llm.to_markdown(
            file_path,
            page_chunks=True,
            use_ocr=True,
            table_strategy="lines",
            show_progress=True,
            header=True,
            footer=True
        )

    except Exception as e:

        print(f"\nNormal PDF extraction failed: {e}")
        print("Trying full OCR...")

        page_data = pymupdf4llm.to_markdown(
            file_path,
            page_chunks=True,
            use_ocr=True,
            force_ocr=True,
            table_strategy="lines",
            show_progress=True
        )


    docs = []

    for page in page_data:

        text = page.get("text", "").strip()

        if not text:
            continue

        metadata = page.get("metadata", {})

        page_number = metadata.get(
            "page_number",
            len(docs) + 1
        )

        docs.append(
            Document(
                page_content=clean_text(text),
                metadata={
                    "source": file_path,
                    "page": page_number
                }
            )
        )

    print(
        f"PDF extraction complete: "
        f"{len(docs)} pages produced searchable text."
    )

    return docs


def load_file(file_path):
    """Load PDF, DOCX or TXT automatically."""

    extension = os.path.splitext(
        file_path
    )[1].lower()

    if extension == ".pdf":

        return load_pdf(file_path)

    elif extension == ".docx":

        docs = Docx2txtLoader(
            file_path
        ).load()

    elif extension == ".txt":

        docs = TextLoader(
            file_path,
            encoding="utf-8"
        ).load()

    else:

        raise ValueError(
            f"Unsupported file type: {extension}"
        )

    for doc in docs:

        doc.page_content = clean_text(
            doc.page_content
        )

        doc.metadata["source"] = file_path

    return docs


def load_urls(urls):

    print("\nLoading URLs...")

    docs = WebBaseLoader(
        web_paths=urls
    ).load()

    for doc in docs:

        doc.page_content = clean_text(
            doc.page_content
        )

    return docs


def build_rag(docs):

    global chunks
    global vector_store
    global tfidf
    global tfidf_matrix


    if not docs:
        raise ValueError(
            "No readable content was found."
        )


    print(
        f"\nLoaded {len(docs)} document sections."
    )


    # Split the extracted content into searchable pieces
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120
    )

    chunks = text_splitter.split_documents(
        docs
    )


    print(
        f"Created {len(chunks)} chunks."
    )


    # Build FAISS semantic search
    print("\nCreating FAISS index...")

    vector_store = FAISS.from_documents(
        chunks,
        embeddings
    )


    # Build keyword search
    chunk_texts = [
        chunk.page_content
        for chunk in chunks
    ]


    tfidf = TfidfVectorizer(
        lowercase=True,
        stop_words="english"
    )


    tfidf_matrix = tfidf.fit_transform(
        chunk_texts
    )


    print("FAISS and keyword search are ready.")


def retrieve_documents(query):

    if vector_store is None:
        raise RuntimeError(
            "No document is loaded."
        )


    # Semantic search
    semantic_results = (
        vector_store.similarity_search_with_score(
            query,
            k=min(15, len(chunks))
        )
    )


    semantic_scores = {}


    for doc, distance in semantic_results:

        try:

            index = next(
                i for i, item in enumerate(chunks)
                if item.page_content == doc.page_content
                and item.metadata == doc.metadata
            )

            semantic_scores[index] = (
                1 / (1 + float(distance))
            )

        except StopIteration:

            continue


    # Keyword search
    query_vector = tfidf.transform(
        [query]
    )

    keyword_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]


    # Combine both retrieval methods
    combined_scores = {}


    for i in range(len(chunks)):

        semantic_score = semantic_scores.get(
            i,
            0
        )

        keyword_score = float(
            keyword_scores[i]
        )

        combined_scores[i] = (
            0.65 * semantic_score +
            0.35 * keyword_score
        )


    ranked_indexes = sorted(
        combined_scores,
        key=combined_scores.get,
        reverse=True
    )


    # List questions need more context
    list_question = any(
        word in query.lower()
        for word in [
            "list",
            "all websites",
            "which websites",
            "what websites",
            "names of",
            "websites are",
            "how many",
            "all"
        ]
    )


    top_k = 15 if list_question else 6

    top_k = min(
        top_k,
        len(chunks)
    )


    return [
        chunks[i]
        for i in ranked_indexes[:top_k]
    ]


system_prompt = """
You are a document question-answering assistant.

Answer the user's question only from the provided document context.

Do not invent information.

For list questions, include all relevant items supported by the
provided context.

Preserve names, URLs, numbers, and technical terms exactly when possible.

When the context contains a PDF page number, cite the page like:
[Page 5]

When the information is not available in the document, say:

"I don't have enough information in the provided document."

Retrieved Context:
{context}

User Query:
{query}

Answer:
"""


def ask_question(user_query):

    retrieved_docs = retrieve_documents(
        user_query
    )


    context_parts = []


    for i, doc in enumerate(
        retrieved_docs,
        1
    ):

        source = doc.metadata.get(
            "source",
            "Unknown source"
        )

        page = doc.metadata.get(
            "page"
        )


        if page:

            location = (
                f"Source: {source} | "
                f"Page: {page}"
            )

        else:

            location = (
                f"Source: {source}"
            )


        context_parts.append(
            f"Document Chunk {i}\n"
            f"{location}\n"
            f"{doc.page_content}"
        )


    context = "\n\n".join(
        context_parts
    )


    final_prompt = system_prompt.format(
        context=context,
        query=user_query
    )


    response = llm.invoke(
        final_prompt
    )


    answer = response.content


    if isinstance(answer, list):

        answer = "\n".join(
            item.get("text", str(item))
            if isinstance(item, dict)
            else str(item)
            for item in answer
        )


    print("\n1. SYSTEM PROMPT")
    print(
        system_prompt
        .replace(
            "{context}",
            "[Retrieved document context]"
        )
        .replace(
            "{query}",
            user_query
        )
    )


    print("\n2. QUERY")
    print(user_query)


    print("\n3. RETRIEVED CONTEXT")

    for i, doc in enumerate(
        retrieved_docs,
        1
    ):

        print(f"\nChunk {i}:")

        if "page" in doc.metadata:

            print(
                f"Page: {doc.metadata['page']}"
            )

        print(
            doc.page_content[:1000]
        )


    print("\n4. ANSWER")
    print(answer)


def load_source():

    print("\nSelect your source:")
    print("1. URLs")
    print("2. Upload PDF / DOCX / TXT")
    print("3. Exit")


    choice = input(
        "\nEnter 1, 2 or 3: "
    ).strip()


    if choice == "1":

        url_input = input(
            "\nEnter URL(s), separated by commas:\n"
        ).strip()


        urls = [
            url.strip()
            for url in url_input.split(",")
            if url.strip()
        ]


        if not urls:

            print(
                "No URLs were entered."
            )

            return False


        docs = load_urls(
            urls
        )


    elif choice == "2":

        print(
            "\nUpload one or more PDF, DOCX or TXT files."
        )


        uploaded = files.upload()


        if not uploaded:

            print(
                "No file was uploaded."
            )

            return False


        docs = []


        for file_path in uploaded.keys():

            print(
                f"\nProcessing: {file_path}"
            )


            try:

                file_docs = load_file(
                    file_path
                )

                docs.extend(
                    file_docs
                )

            except Exception as e:

                print(
                    f"Could not load {file_path}: {e}"
                )


    elif choice == "3":

        return None


    else:

        print(
            "Invalid choice."
        )

        return False


    if not docs:

        print(
            "No readable document content found."
        )

        return False


    build_rag(
        docs
    )


    return True


# Load the first source
status = load_source()


if status is None:

    print("\nExited.")


elif status:

    # Keep running until the user explicitly chooses Exit
    while True:

        print(
            "\n" + "=" * 60
        )

        print(
            "RAG is ready. Ask a question."
        )

        print(
            "Type 'menu' to load another document or exit."
        )


        user_query = input(
            "\nEnter your question: "
        ).strip()


        if not user_query:

            continue


        if user_query.lower() == "menu":

            status = load_source()


            if status is None:

                print(
                    "\nExited."
                )

                break


            continue


        try:

            ask_question(
                user_query
            )

        except Exception as e:

            print(
                f"\nError while answering: {e}"
            )


        print(
            "\nWhat would you like to do next?"
        )

        print(
            "1. Ask another question"
        )

        print(
            "2. Load a new document / URL"
        )

        print(
            "3. Exit"
        )


        next_action = input(
            "\nEnter 1, 2 or 3: "
        ).strip()


        if next_action == "1":

            continue


        elif next_action == "2":

            status = load_source()


            if status is None:

                print(
                    "\nExited."
                )

                break


            continue


        elif next_action == "3":

            print(
                "\nRAG session ended."
            )

            break


        else:

            print(
                "\nInvalid choice. Continuing..."
            )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 30

/tmp/ipykernel_2905/3995832553.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


Enter your Google API key: ··········

Select your source:
1. URLs
2. Upload PDF / DOCX / TXT
3. Exit

Enter 1, 2 or 3: 1

Enter URL(s), separated by commas:
https://en.wikipedia.org/wiki/Return_policy,https://en.wikipedia.org/wiki/Return_merchandise_authorization

Loading URLs...

Loaded 2 document sections.
Created 25 chunks.

Creating FAISS index...
FAISS and keyword search are ready.

RAG is ready. Ask a question.
Type 'menu' to load another document or exit.

Enter your question: how do i return my orders



1. SYSTEM PROMPT

You are a document question-answering assistant.

Answer the user's question only from the provided document context.

Do not invent information.

For list questions, include all relevant items supported by the
provided context.

Preserve names, URLs, numbers, and technical terms exactly when possible.

When the context contains a PDF page number, cite the page like:
[Page 5]

When the information is not available in the document, say:

"I don't have enough information in the provided document."

Retrieved Context:
[Retrieved document context]

User Query:
how do i return my orders

Answer:


2. QUERY
how do i return my orders

3. RETRIEVED CONTEXT

Chunk 1:
The issuance of an RMA is a key gatekeeping point in the reverse logistics cycle, providing the vendor with a final opportunity to diagnose and correct the customer's problem with the product. The reasons for a product return vary and include improper installation by the customer or inability to configure the pro

KeyboardInterrupt: Interrupted by user